### クラスタリング分離具合の図示

全ての２つの説明変数組み合わせに対してクラスタリングを行い、分離具合を確認します。


In [ ]:
import pandas as pd
g_df = pd.read_csv(
    "../data_calculated/Carbon8_descriptor.csv", index_col=[0, 1])
g_df


次のスクリプトは時間がかかるので予め図を作成してあります。図があると実行されませんので，
実行する場合はcluster_images/フォルダにある図を削除してください。

In [ ]:
print("all done")


In [ ]:
from itertools import combinations
from sklearn.mixture import GaussianMixture
import collections
import sklearn.cluster
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
%matplotlib inline

"""
3. data analysis
"""


def gmm2D_predict(X_, n_components, random_state, title=None, filename=None):
    """ plot clusters in the descriptor space

    Args:
        X_ (np.array): descriptor
        n_components (int): the number of clusters
        random_state (int): random state for GMM clustering
        title (str, optional): title of the figure. Defaults to None.
        filename (str, optional): filename. Defaults to None.

    Returns:
        GaussianMixture: gaussian mixture model instance
        collections.Counter: the number of samples in clusters
    """

    X = X_.copy()

    fig, axes = plt.subplots(1,2)
    colors = ["b", "r", "y", "m", "c"]
    ax = axes[0]
    sns.kdeplot(x=X[:, 0], y=X[:, 1], ax=ax)

    ax = axes[1]
    sns.kdeplot(x=X[:, 0], y=X[:, 1], fill=True, ax=ax)

    clf = GaussianMixture(
        n_components=n_components, random_state=random_state)
    clf.fit(X)
    print("means", clf.means_)
    print("covariance", clf.covariances_)
    yp = clf.predict(X)
    count = collections.Counter(yp)

    for i in range(n_components):
        data = []
        for x1, y1 in zip(X_, yp):
            if y1 == i:
                data.append(x1)
        data = np.array(data)
        if data.shape[0] > 0:
            ax.plot(data[:, 0], data[:, 1], colors[i]+".", label=str(i))

    ax.plot(clf.means_[:, 0], clf.means_[:, 1], "o")
    ax.legend()
    if title is not None:
        ax.set_title(title)
    fig.tight_layout()
    if filename is not None:
        fig.savefig(filename)
        print("saved to", filename)
    fig.show()

    return clf, count


"""
4. visualiztion
"""


def make_gmm2D_plot2(X_, pairs, n_components, random_state):
    """helper routine for gmm2D_predict

    Args:
        X_ (np.array): descriptor
        pairs (list): a list of feature pairs
        n_components (int): the number of clusters
        random_state (int): random state
    """
    for i, j in pairs:
        print("pairs", i, j)
        vallist = []
        for ncl in n_components:
            print("cluster", ncl)
            filename = "cluster_images/{}_{}_{}.png".format(i, j, ncl)
            if os.path.exists(filename):
                print("filename", filename, "exists. break")
                break
            title = "({},{}), n_components={}".format(i, j, ncl)
            clf, count = gmm2D_predict(
                X_[:, [i, j]], ncl, random_state, title=title, filename=filename)
            vallist.append([i, j, ncl, count])
        for iv, x in enumerate(vallist):
            if iv == 0:
                print(iv == 0, x)
            else:
                print(iv == 0, x, vallist[iv][3]-vallist[iv-1][3])


def try_gmm(df):
    """plot feature pairs

    Args:
        df (pd.DataFrame): data
    """
    X = df.values
    labels = df.columns
    pairs = list(combinations(range(15), 2))
    for i in range(1):
        make_gmm2D_plot2(X, pairs=pairs, n_components=[3], random_state=i)
        # make_gmm2D_plot2(X,pairs=[[0,1]],n_components=[3,4,5],random_state=i)


try_gmm(g_df)

予めimage fileが作成されています。imageをご覧ください。

jupterからは以下のようにして見ることができます。

In [ ]:
from IPython.display import Image, display_png
display_png(Image('cluster_images/0_6_3.png'))
